# AWS Bedrock Managed Model Evaluation API

**Reference:** Slide 40 (*Hands-On RAG with AWS*)

This notebook demonstrates automated RAG evaluation using **AWS Bedrock's fully managed Model Evaluation service** (`boto3.client('bedrock').create_evaluation_job`), assessing built-in automated metrics:
- **Faithfulness (Groundedness)**: Checks if assertions in the generated response are factually grounded in the retrieved context chunks.
- **Answer Relevance**: Measures how directly the generated answer satisfies the user's initial question.
- **Context Relevance (Retrieval Precision)**: Measures whether the retrieved chunks are relevant to the query or contain distracting noise.

We also compare AWS Managed Evaluation against client-side evaluation libraries like **RAGAS**.

In [ ]:
import os
import sys
import time
import pandas as pd
from bedrock_managed_evaluation import BedrockManagedEvaluator, compare_evaluation_frameworks

# Initialize managed evaluator (auto-detects credentials or falls back to mock mode)
evaluator = BedrockManagedEvaluator()
print(f"Evaluator initialized. Mock Mode: {evaluator.mock_mode}")

## 1. Prepare Standard Evaluation Dataset (JSONL)
Bedrock Model Evaluation requires a JSONL dataset with `prompt`, `referenceResponse`, and `retrievedContext`.

In [ ]:
dataset_file = evaluator.create_rag_dataset(output_path="demo_rag_dataset.jsonl")

# Inspect first record
with open(dataset_file) as f:
    print("First Sample Record:")
    print(f.readline())

## 2. Trigger Automated Evaluation Job (`create_evaluation_job`)
We specify the model under test (Claude 3 Haiku), the judge model (Claude 3.5 Sonnet), and the built-in RAG metrics.

In [ ]:
job_name = f"rag-managed-eval-{int(time.time())}"
job_response = evaluator.trigger_evaluation_job(
    job_name=job_name,
    dataset_s3_uri=f"s3://evaluation-bucket/{dataset_file}",
    model_id="anthropic.claude-3-haiku-20240307-v1:0",
    judge_model_id="anthropic.claude-3-5-sonnet-20241022-v2:0",
    metrics=["Faithfulness", "AnswerRelevance", "ContextRelevance"]
)
print(f"Evaluation Job Submitted: {job_response.get('jobArn')}")

## 3. Poll Status & Inspect RAG Metrics (`get_evaluation_job`)
Poll the job status and inspect computed metric scores.

In [ ]:
status = evaluator.get_job_status(job_name)
print(f"Job Status: {status.get('status')}")

if "evaluationResults" in status:
    print("\n--- Evaluated RAG Metrics ---")
    for metric, score in status["evaluationResults"].items():
        print(f"{metric:<20}: {score:.2f} ({score * 100:.1f}%)")

## 4. Architectural Comparison: Bedrock Managed Evaluation vs. RAGAS
When should you choose Bedrock's native evaluation over client-side libraries like RAGAS?

In [ ]:
comparison = compare_evaluation_frameworks()
df_comp = pd.DataFrame(comparison)
df_comp

## Cleanup

In [ ]:
if os.path.exists(dataset_file):
    os.remove(dataset_file)
print("Cleaned up temporary dataset file.")